In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass


def logistic_step(x, r):
    """One update; use x in [0, 1] and r in [0, 4]."""
    return r * x * (1 - x)

def cobweb_points_up_to(r, frame, x0=0.1):
    """Return path, last completed state, and completed iterations.

    Frame 0 is (x0, 0). Each iteration has two movements:
    vertical to the curve, then horizontal to the diagonal.
    """
    if not 0 <= r <= 4 or not 0 <= x0 <= 1:
        raise ValueError("Use r in [0, 4] and x0 in [0, 1].")
    if not isinstance(frame, (int, np.integer)) or frame < 0:
        raise ValueError("frame must be a nonnegative integer.")
    x = float(x0)
    points = [(x, 0.0)]
    for _ in range(frame // 2):
        next_x = logistic_step(x, r)
        points.extend([(x, next_x), (next_x, next_x)])
        x = next_x
    if frame % 2:
        points.append((x, logistic_step(x, r)))
    return np.asarray(points), x, frame // 2


def prepare_bifurcation(r_values, x0=0.1, burn_in=12000, n_keep=192):
    """Independent trajectories, with the transient discarded in each column."""
    x = np.full(len(r_values), float(x0))
    for _ in range(burn_in):
        x = logistic_step(x, r_values)
    late = np.empty((len(r_values), n_keep))
    for j in range(n_keep):
        x = logistic_step(x, r_values)
        late[:, j] = x
    return late


def show_cobweb_bifurcation():
    x0, iterations = 0.1, 40
    r_values = np.linspace(2.5, 4.0, 1501)
    late = prepare_bifurcation(r_values, x0=x0)
    bif_x = np.repeat(r_values, late.shape[1])
    bif_y = late.ravel()

    sweep = widgets.Play(value=0, min=0, max=len(r_values)-1,
                         step=5, interval=100, repeat=False)
    parameter = widgets.IntSlider(value=0, min=0, max=sweep.max,
                                  description="r position:", readout=False,
                                  continuous_update=False,
                                  layout=widgets.Layout(width="420px"))
    r_readout = widgets.HTML()
    reset = widgets.Button(description="Reset sweep")
    output = widgets.Output()
    links = [widgets.link((sweep, "value"), (parameter, "value"))]

    def draw(change=None):
        i = parameter.value
        r = r_values[i]
        points, _, completed = cobweb_points_up_to(r, 2*iterations, x0)
        r_readout.value = f"<b>r = {r:.3f}</b>"
        with output:
            clear_output(wait=True)
            fig, (top, bottom) = plt.subplots(
                2, 1, figsize=(8, 10), gridspec_kw={"height_ratios": [1.2, 1]},
                constrained_layout=True,
            )
            xs = np.linspace(0, 1, 400)
            top.plot(xs, logistic_step(xs, r), color="black", label="The rule")
            top.plot(xs, xs, "--", color="0.6", label="$x_{t+1}=x_t$")
            top.plot(points[:, 0], points[:, 1], color="tab:red", lw=1)
            top.scatter(*points[-1], color="tab:red", s=30, zorder=4)
            top.set(xlim=(0, 1), ylim=(0, 1), xlabel="$x_t$", ylabel="$x_{t+1}$",
                    title=f"Cobweb | r = {r:.3f}"
                    # title=f"Cobweb | r = {r:.3f} | {completed} completed iterations"
                    )
            top.set_aspect("equal", adjustable="box")
            top.legend(loc="upper left", fontsize=9)

            # Include every sampled r through the current position, even if
            # playback skips positions. Scrubbing backward removes later columns.
            end = (i+1)*late.shape[1]
            bottom.plot(bif_x[:end], bif_y[:end], ",", color="#087f83", alpha=0.45)
            bottom.axvline(r, color="0.6", lw=0.8)
            bottom.scatter(np.full(late.shape[1], r), late[i],
                           color="tab:orange", s=7, alpha=0.7, zorder=3)
            bottom.set(xlim=(2.5, 4.0), ylim=(0, 1), xlabel="Parameter r",
                       ylabel="Attractor value (post-transient state)", title="Building the bifurcation diagram")
            bottom.text(0.02, 0.97, "Orange: current r", transform=bottom.transAxes,
                        va="top", color="#b45709")
            plt.show()
            plt.close(fig)

    def reset_sweep(button):
        sweep.playing = False
        parameter.value = 0

    parameter.observe(draw, names="value")
    reset.on_click(reset_sweep)
    display(widgets.VBox([
        widgets.HTML("<b>Sweep r to build the diagram</b>"),
        widgets.HBox([sweep, reset, r_readout]), parameter,
        widgets.HTML("Both panels start at x₀ = 0.1."
                    #  "Top: first 40 iterations. "
                    #  "Bottom: 192 observations after 12,000 settling updates. "
                    #  "The lower panel is computed separately; it is not inferred "
                    #  "from the short cobweb. Finite settling may leave transients "
                    #  "near bifurcations."
                     ), output,
    ]))
    draw()
    return links


animation_links = show_cobweb_bifurcation()
